# FullStack IA · Apprentissage automatique

**Objectif :** Prétraitement, régressions, régularisation, arbres, boosting, séries temporelles et clustering.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Le vocabulaire du machine learning

Le **machine learning** consiste à faire apprendre à un programme une règle à partir d'exemples, au lieu de l'écrire à la main.

## Les grandes familles

- **Apprentissage supervisé** : on dispose d'exemples avec la bonne réponse (la **cible**, ou *target*, notée y).
  - **Régression** : la cible est un nombre (un prix, une masse, une durée).
  - **Classification** : la cible est une catégorie (spam / non spam, espèce, survie).
- **Apprentissage non supervisé** : pas de cible. On cherche une structure : des groupes (*clustering*), une réduction de dimensions.
- L'**IA générative** produit du contenu (texte, images, code). Les grands modèles de langage sont entraînés de façon auto-supervisée : prédire le mot suivant.

## Features et cible

Les **features** (caractéristiques, notées X) sont les colonnes qui servent à prédire. La **cible** y est la colonne à prédire.

In [ ]:
import pandas as pd
df = pd.read_csv("penguins.csv").dropna()
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]]   # matrice : majuscule
y = df["species"]                                                   # vecteur : minuscule
print(X.shape, y.shape)

## Séparer entraînement et test

*Animation interactive « split » : à voir sur la plateforme web.*

Un modèle doit être évalué sur des données qu'il **n'a jamais vues**, sinon on mesure sa mémoire et non sa capacité à généraliser.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "exemples d'entraînement,", len(X_test), "de test")

`random_state` rend le découpage reproductible. `stratify=y` garde les mêmes proportions de classes dans les deux parties.

## Sous-apprentissage et surapprentissage

*Animation interactive « overfit » : à voir sur la plateforme web.*

- **Sous-apprentissage** (*underfitting*) : le modèle est trop simple. Il se trompe autant à l'entraînement qu'au test.
- **Surapprentissage** (*overfitting*) : le modèle a appris par cœur les données d'entraînement, bruit compris. Excellent à l'entraînement, mauvais au test.
- Le bon modèle se situe entre les deux : c'est le **compromis biais-variance**.

## Toujours une baseline

Avant tout modèle, calcule le score d'une stratégie naïve : prédire la classe majoritaire, ou la moyenne. Un modèle qui ne fait pas mieux ne sert à rien.

In [ ]:
print(y_train.value_counts(normalize=True).round(3))

> **À retenir** : features X, cible y, séparation train/test, baseline. Ces quatre réflexes s'appliquent à tous les projets, y compris en deep learning.

### Exercice 1 · Préparer X et y

Avec `tips.csv`, on veut prédire le pourboire. Crée `X` (colonnes `total_bill` et `size`) et `y` (colonne `tip`), puis sépare-les avec `train_test_split` (25 % de test, `random_state=0`) en `X_train, X_test, y_train, y_test`.

<details><summary>Indice</summary>

X avec des doubles crochets, y avec des simples. train_test_split(X, y, test_size=0.25, random_state=0).
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
tips = pd.read_csv("tips.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert list(X.columns) == ["total_bill", "size"], f"Colonnes de X : {list(X.columns)}."
assert X_train.shape == (183, 2) and X_test.shape == (61, 2), f"Formes : {X_train.shape} et {X_test.shape}, j'attendais (183, 2) et (61, 2)."
assert len(y_train) == 183, "y_train doit avoir 183 valeurs."
print("Bravo, tout est correct.")

### Exercice 2 · Le score à battre

Sur le Titanic, calcule `baseline` : l'exactitude (*accuracy*) obtenue en prédisant **toujours** la classe la plus fréquente de `survived`. Exprime-la entre 0 et 1.

<details><summary>Indice</summary>

value_counts(normalize=True) donne les proportions ; la plus grande est l'exactitude de la stratégie « toujours la classe majoritaire ».
</details>

In [ ]:
import pandas as pd
t = pd.read_csv("titanic.csv")

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(baseline - 0.6162) < 0.001, f"baseline vaut {baseline}, j'attendais environ 0.616."
print("Bravo, tout est correct.")

---
## Classification et métriques

La **régression logistique** est le modèle de classification de base. Malgré son nom, elle classe : elle calcule une combinaison linéaire des features, puis la transforme en probabilité entre 0 et 1 grâce à la fonction **sigmoïde**.

$$
P(y = 1 \mid x) = \sigma(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}
$$

## Préparer les données

Les modèles ne comprennent que des nombres. Il faut **encoder** les catégories et **imputer** les valeurs manquantes.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)          # encodage binaire
t["age"] = t["age"].fillna(t["age"].median())
features = ["pclass", "femme", "age", "fare", "sibsp", "parch"]
X, y = t[features], t["survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]       # probabilité de survie
print(pred[:5], proba[:5].round(2))

Pour une variable à plusieurs catégories (comme `embarked`), on utilise le **one-hot encoding** : `pd.get_dummies(t["embarked"])` crée une colonne 0/1 par catégorie.

## La matrice de confusion

In [ ]:
from sklearn.metrics import confusion_matrix
pd.DataFrame(confusion_matrix(y_test, pred),
             index=["réel : décédé", "réel : survivant"],
             columns=["prédit : décédé", "prédit : survivant"])

- **Vrais positifs (VP)** : survivants prédits survivants. **Faux positifs (FP)** : décédés prédits survivants.
- **Vrais négatifs (VN)** et **faux négatifs (FN)** : de même pour l'autre classe.

## Les métriques

$$
\text{précision} = \frac{VP}{VP + FP} \qquad \text{rappel} = \frac{VP}{VP + FN} \qquad F_1 = 2 \cdot \frac{\text{précision} \cdot \text{rappel}}{\text{précision} + \text{rappel}}
$$

- **Exactitude** (*accuracy*) : la part de bonnes réponses. Trompeuse si les classes sont déséquilibrées.
- **Précision** : parmi les prédits positifs, combien le sont vraiment ? Importante quand un faux positif coûte cher (un mail légitime classé spam).
- **Rappel** (*recall*) : parmi les vrais positifs, combien en a-t-on trouvé ? Important quand un faux négatif coûte cher (une fraude ou une maladie non détectée).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print(f"exactitude {accuracy_score(y_test, pred):.3f}  précision {precision_score(y_test, pred):.3f}  "
      f"rappel {recall_score(y_test, pred):.3f}  F1 {f1_score(y_test, pred):.3f}")

> **À retenir** : choisis ta métrique en fonction du **coût des erreurs** dans le problème métier, et dis-le explicitement dans tes projets.

### Exercice 1 · Ajouter le port d'embarquement

Améliore le modèle de la leçon : ajoute un encodage one-hot de `embarked` (après avoir remplacé les valeurs manquantes par `"S"`). Construis `X` avec les features de base plus ces colonnes, entraîne `clf` (même split que la leçon) et calcule `acc`, l'exactitude sur le test. Elle doit dépasser 0.78.

<details><summary>Indice</summary>

pd.get_dummies(t["embarked"].fillna("S"), prefix="port", dtype=int), puis pd.concat([t[base], ports], axis=1).
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)
t["age"] = t["age"].fillna(t["age"].median())
base = ["pclass", "femme", "age", "fare", "sibsp", "parch"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert X.shape[1] == 9, f"X a {X.shape[1]} colonnes, j'en attendais 9 (6 de base + 3 ports)."
assert X.isna().sum().sum() == 0, "X contient encore des valeurs manquantes."
assert acc > 0.78, f"acc vaut {acc:.3f}, vise plus de 0.78."
print("Bravo, tout est correct.")

### Exercice 2 · Précision et rappel à la main

À partir de la matrice de confusion ci-dessous (lignes = réel, colonnes = prédit, ordre [négatif, positif]), calcule `precision`, `rappel` et `f1` **sans scikit-learn**.

<details><summary>Indice</summary>

cm.ravel() donne [VN, FP, FN, VP]. Applique ensuite les formules.
</details>

In [ ]:
import numpy as np
cm = np.array([[95, 15],
               [20, 49]])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(precision - 49 / 64) < 1e-9, f"precision vaut {precision}, j'attendais 49/64 ≈ 0.766."
assert abs(rappel - 49 / 69) < 1e-9, f"rappel vaut {rappel}, j'attendais 49/69 ≈ 0.710."
assert abs(f1 - 0.7368) < 1e-3, f"f1 vaut {f1}."
print("Bravo, tout est correct.")

---
## Régression avec scikit-learn

**scikit-learn** propose des dizaines de modèles avec la même interface : on crée le modèle, on appelle `fit` sur les données d'entraînement, puis `predict`. Tu as codé cette interface toi-même en semaine 3.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv("penguins.csv").dropna()
X = df[["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]]
y = df["body_mass_g"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele = LinearRegression()
modele.fit(X_train, y_train)
print("coefficients :", modele.coef_.round(2))
print("biais :", round(modele.intercept_, 1))

Chaque coefficient indique de combien de grammes la masse prédite augmente quand la variable augmente d'une unité, les autres restant fixes.

## Évaluer une régression

- **MAE** (erreur absolue moyenne) : l'erreur moyenne, dans l'unité de la cible. Facile à expliquer.
- **RMSE** (racine de l'erreur quadratique moyenne) : pénalise davantage les grosses erreurs.
- **R²** : la part de la variance expliquée par le modèle. 1 = parfait, 0 = pas mieux que prédire la moyenne.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

pred = modele.predict(X_test)
print(f"MAE  : {mean_absolute_error(y_test, pred):.0f} g")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.0f} g")
print(f"R²   : {r2_score(y_test, pred):.3f}")

Compare toujours à la baseline :

In [ ]:
baseline = np.full(len(y_test), y_train.mean())
print(f"RMSE baseline : {np.sqrt(mean_squared_error(y_test, baseline)):.0f} g")

## Visualiser les erreurs

In [ ]:
import plotly.express as px
resultats = pd.DataFrame({"réel": y_test, "prédit": pred})
px.scatter(resultats, x="réel", y="prédit", title="Prédictions contre valeurs réelles (g)")

Plus les points sont proches de la diagonale, meilleur est le modèle.

> **À retenir** : on ne calcule les métriques que sur le jeu de **test**. Un score mesuré sur les données d'entraînement est toujours trop optimiste.

### Exercice 1 · Prédire le pourboire

Entraîne une `LinearRegression` nommée `modele` qui prédit `tip` à partir de `total_bill` et `size` (split 25 %, `random_state=0`). Calcule `mae` sur le jeu de test et `mae_baseline`, la MAE obtenue en prédisant toujours la moyenne de `y_train`.

<details><summary>Indice</summary>

modele = LinearRegression().fit(X_train, y_train), puis mean_absolute_error(y_test, modele.predict(X_test)).
</details>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

tips = pd.read_csv("tips.csv")
X = tips[["total_bill", "size"]]
y = tips["tip"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.linear_model import LinearRegression
assert isinstance(modele, LinearRegression), "modele doit être une LinearRegression."
assert hasattr(modele, "coef_"), "Le modèle n'est pas entraîné : appelle fit."
assert mae < mae_baseline, "Ton modèle devrait battre la baseline."
assert abs(mae - 0.6889) < 0.01, f"mae vaut {mae:.4f}, j'attendais environ 0.689."
print("Bravo, tout est correct.")

---
## Surapprentissage et validation croisée

Un **arbre de décision** pose une suite de questions (« sexe = femme ? », « classe ≤ 2 ? ») pour classer. Plus il est profond, plus il peut coller aux données d'entraînement, jusqu'à les apprendre par cœur.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)
t["age"] = t["age"].fillna(t["age"].median())
X = t[["pclass", "femme", "age", "fare", "sibsp", "parch"]]
y = t["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lignes = []
for profondeur in [1, 2, 3, 5, 8, 12, None]:
    arbre = DecisionTreeClassifier(max_depth=profondeur, random_state=0).fit(X_train, y_train)
    lignes.append({"profondeur": str(profondeur), "train": arbre.score(X_train, y_train),
                   "test": arbre.score(X_test, y_test)})
pd.DataFrame(lignes).round(3)

Sans limite de profondeur, l'arbre atteint presque 100 % sur l'entraînement mais pas sur le test : c'est le **surapprentissage**. L'écart entre les deux scores est le signal d'alarme.

## La validation croisée

Un seul découpage train/test dépend du hasard. La **validation croisée à k plis** (*k-fold*) découpe les données en k parts, entraîne k fois en gardant à chaque fois une part différente pour l'évaluation, puis fait la moyenne.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X, y, cv=5)
print(scores.round(3), "→ moyenne", scores.mean().round(3), "± écart-type", scores.std().round(3))

## Choisir un hyperparamètre

`max_depth` est un **hyperparamètre** : on le fixe avant l'entraînement, il n'est pas appris. On le choisit par validation croisée, **jamais** sur le jeu de test, qui doit rester intact jusqu'à l'évaluation finale.

> **À retenir** : le jeu de test est comme l'examen final. Si tu t'en sers pour régler ton modèle, ton score final n'est plus fiable.

### Exercice 1 · Trouver la meilleure profondeur

Pour chaque `max_depth` de 1 à 10, calcule le score moyen en validation croisée (`cv=5`, `random_state=0` pour l'arbre) sur `X, y`. Stocke les résultats dans le dictionnaire `scores_cv` `{profondeur: score moyen}`, puis `meilleure_profondeur`.

<details><summary>Indice</summary>

Une boucle for d in range(1, 11), puis max(scores_cv, key=scores_cv.get).
</details>

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

t = pd.read_csv("titanic.csv")
t["femme"] = (t["sex"] == "female").astype(int)
t["age"] = t["age"].fillna(t["age"].median())
X = t[["pclass", "femme", "age", "fare", "sibsp", "parch"]]
y = t["survived"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
ref = {d: cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X, y, cv=5).mean() for d in range(1, 11)}
assert set(scores_cv) == set(range(1, 11)), "scores_cv doit avoir les clés 1 à 10."
assert meilleure_profondeur == max(ref, key=ref.get), f"meilleure_profondeur vaut {meilleure_profondeur}, j'attendais {max(ref, key=ref.get)}."
print("Bravo, tout est correct.")

---
## Projet final · Classifier les espèces de manchots

Ton premier projet de portfolio complet : à partir des mesures d'un manchot, prédire son espèce (Adelie, Chinstrap ou Gentoo). Tu mobilises toute la mise à niveau : nettoyage, exploration, modèle, évaluation, et présentation.

Une **pipeline** scikit-learn enchaîne les étapes de préparation et le modèle en un seul objet. Elle évite les erreurs, en particulier les fuites de données : la standardisation n'apprend que sur le jeu d'entraînement.

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modele.fit(X_train, y_train)
```

Modèle de README pour GitHub (à rédiger en anglais) :

```markdown
# Penguin species classifier
## Problem       What we predict and why it matters
## Data          Palmer Penguins: 344 rows, 7 columns, cleaning choices
## Approach      Features, model, train/test split
## Results       Accuracy, confusion matrix, baseline comparison
## How to run    pip install -r requirements.txt, then open the notebook
```

**À faire :**

- [ ] Refaire le projet dans un notebook sur ton ordinateur
- [ ] Ajouter un graphique exploratoire (nageoire contre bec, coloré par espèce)
- [ ] Comparer avec un arbre de décision et expliquer ton choix
- [ ] Écrire le README en anglais à partir du modèle ci-dessus
- [ ] Publier sur GitHub et épingler le dépôt sur ton profil

### Exercice 1 · Préparer les données

Charge `penguins.csv`, supprime les lignes incomplètes, et crée `X` avec les 4 mesures numériques plus une colonne `male` (1 si `sex == "MALE"`, sinon 0), et `y` = `species`. Sépare en train/test (20 %, `random_state=42`, stratifié).

<details><summary>Indice</summary>

dropna(), puis (df["sex"] == "MALE").astype(int). N'oublie pas stratify=y.
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert X.shape == (333, 5), f"X a la forme {X.shape}, j'attendais (333, 5)."
assert "male" in X.columns and set(X["male"].unique()) == {0, 1}, "La colonne male doit contenir des 0 et des 1."
assert len(X_test) == 67, f"X_test a {len(X_test)} lignes, j'en attendais 67."
print("Bravo, tout est correct.")

### Exercice 2 · Entraîner et évaluer

Crée `modele`, une pipeline `StandardScaler` + `LogisticRegression(max_iter=1000)`, entraîne-la, calcule `acc` sur le test et `cm`, la matrice de confusion. Vise une exactitude d'au moins 0.97.

<details><summary>Indice</summary>

make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)) s'utilise comme un modèle : fit puis predict.
</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("penguins.csv").dropna()
df["male"] = (df["sex"] == "MALE").astype(int)
X = df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "male"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert hasattr(modele, "predict") and hasattr(modele, "steps"), "modele doit être une pipeline (make_pipeline)."
assert acc >= 0.97, f"acc vaut {acc:.3f}, vise au moins 0.97."
assert np.asarray(cm).shape == (3, 3), "cm doit être une matrice 3 × 3 (une ligne et une colonne par espèce)."
print("Bravo, tout est correct.")

### Exercice 3 · Prédire un nouveau manchot

Utilise ta pipeline (recopie le code nécessaire) pour prédire l'espèce d'un manchot mâle avec un bec de 50 mm de long et 15 mm d'épaisseur, une nageoire de 220 mm et une masse de 5 500 g. Stocke le nom de l'espèce dans `espece` et la probabilité associée dans `confiance`.

<details><summary>Indice</summary>

Le DataFrame doit avoir exactement les mêmes colonnes que X. predict(...)[0] donne l'espèce, predict_proba(...).max() la probabilité.
</details>

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("penguins.csv").dropna()
df["male"] = (df["sex"] == "MALE").astype(int)
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "male"]
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(df[cols], df["species"])

nouveau = pd.DataFrame([{
    # à compléter
}])

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert espece == "Gentoo", f"espece vaut {espece!r}."
assert 0.9 < confiance <= 1, f"confiance vaut {confiance}, elle devrait être une probabilité élevée."
print("Bravo, tout est correct.")